# Tutorial 16b: Balanced geometry-domain transfer learning

This tutorial uses only the expanded `GeneralizedCapNInterdigital` dataset. We will:

1. connect every swept geometry variable to the static-v0 embedding and six Q3D capacitances;
2. compare finger-count specialists with coverage- and budget-matched generalists;
3. adapt a foundation model trained at one finger count and measure when transfer helps;
4. turn embedding similarity into an **applicability score** that can route uncertain designs back to simulation.

The goal is not to manufacture a universal transfer-learning win. It is to establish where transfer is useful,
where it fails, and how the embedding can tell us the difference.

> **Controlled-cohort companion to Tutorial 16.** This notebook repeats the same experiment, model, split policy, statistical analysis, ablations, and visualizations after deterministically limiting every finger-count domain to exactly **1,260 designs**. The only scientific variable changed is domain size.

In [1]:
import hashlib
import json
import logging
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots
from scipy.stats import spearmanr, t as student_t, ttest_1samp

from squadds.layouts import (
    SHAPE_SIZE,
    TransferRidgeRegressor,
    V0KernelFeatureProjector,
    canonical_design_id,
    compress_v0_embeddings,
    regression_scores,
)

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
logging.getLogger("httpx").setLevel(logging.WARNING)

RELEASE_ROOT = os.getenv("SQUADDS_TUTORIAL16_RELEASE_ROOT")
DATABASE_REVISION_OVERRIDE = os.getenv("SQUADDS_DATABASE_REVISION")
EMBEDDING_REVISION_OVERRIDE = os.getenv("SQUADDS_EMBEDDING_REVISION")
DATABASE_REVISION = DATABASE_REVISION_OVERRIDE or "main"
EMBEDDING_REVISION = EMBEDDING_REVISION_OVERRIDE or "main"
EXPANDED_DATABASE_REVISION = "refs/pr/23"
EXPANDED_EMBEDDING_REVISION = "refs/pr/2"
EXPECTED_EXPANDED_ROWS = 20_000
BALANCED_ROWS_PER_DOMAIN = 1_260
EXPECTED_BALANCED_ROWS = 13 * BALANCED_ROWS_PER_DOMAIN
CACHE_ROOT = Path(os.getenv(
    "SQUADDS_TUTORIAL16B_CACHE",
    Path.home() / ".cache" / "squadds" / "tutorial16b",
))
FRACTIONS = [percentage / 100 for percentage in range(1, 101)]
TARGET_COLUMNS = [
    "C_NS_fF", "C_NG_fF", "C_SG_fF",
    "C_NN_fF", "C_SS_fF", "C_GG_fF",
]
ML_TARGETS = ["C_NS_fF", "C_NG_fF", "C_SG_fF"]
BASE_FINGER_COUNT = 8
RANDOM_SEED = 16
N_REPEATS = 12
CONFIDENCE_LEVEL = 0.95

if RELEASE_ROOT:
    release_root = Path(RELEASE_ROOT)
    database_path = release_root / "db/coupler-GeneralizedCapNInterdigital-cap_matrix.json"
    embedding_path = release_root / "embeddings/metadata/static-embedding-v0.parquet"
    database_source = embedding_source = f"local: {release_root}"
else:
    database_path = Path(hf_hub_download(
        "SQuADDS/SQuADDS_DB",
        "coupler-GeneralizedCapNInterdigital-cap_matrix.json",
        repo_type="dataset",
        revision=DATABASE_REVISION,
    ))
    database_rows = len(json.loads(database_path.read_text()))
    if database_rows < EXPECTED_EXPANDED_ROWS and DATABASE_REVISION_OVERRIDE is None:
        DATABASE_REVISION = EXPANDED_DATABASE_REVISION
        database_path = Path(hf_hub_download(
            "SQuADDS/SQuADDS_DB",
            "coupler-GeneralizedCapNInterdigital-cap_matrix.json",
            repo_type="dataset",
            revision=DATABASE_REVISION,
        ))
        if EMBEDDING_REVISION_OVERRIDE is None:
            EMBEDDING_REVISION = EXPANDED_EMBEDDING_REVISION
    embedding_path = Path(hf_hub_download(
        "SQuADDS/SQuADDS_Layout_Embeddings",
        "metadata/static-embedding-v0.parquet",
        repo_type="dataset",
        revision=EMBEDDING_REVISION,
    ))
    database_source = f"Hugging Face revision: {DATABASE_REVISION}"
    embedding_source = f"Hugging Face revision: {EMBEDDING_REVISION}"

release_status = pd.DataFrame({
    "artifact": ["simulation database", "static-v0 embeddings"],
    "source": [database_source, embedding_source],
    "local_path": [str(database_path), str(embedding_path)],
})
release_status

,artifact,source,local_path
0,simulation database,Hugging Face revision: main,/Users/shanto/.cache/huggingface/hub/datasets-...
1,static-v0 embeddings,Hugging Face revision: main,/Users/shanto/.cache/huggingface/hub/datasets-...


## 1. Explore the geometry and capacitance spaces

The public simulation table is the semantic source of truth: it contains design options and capacitance
targets in **fF**. The embedding table is the geometric source of truth. We join them by `design_id`.

To keep memory bounded, the loader streams 64 embeddings at a time. It retains a 155-dimensional compact
v0 view (parameter sum + 10 moments + a pooled 12×12 shape), while a randomized sketch uses every one of
the 9,227 v0 dimensions for the two-dimensional map.

Before calculating any embedding projection or statistic, we select the same number of designs from each of the 13 exact finger-count domains. A fixed seed and stable design-ID ordering make the 1,260-row samples reproducible. Thus each domain contributes equally to every figure and experiment below.

In [2]:
def numeric_option(value):
    if isinstance(value, (int, float, bool)):
        return float(value)
    if isinstance(value, str):
        match = re.fullmatch(
            r"\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*(?:um|nm|mm|deg)?\s*",
            value,
        )
        return float(match.group(1)) if match else None
    return None


database_rows = json.loads(database_path.read_text())
simulation_records = []
for row in database_rows:
    options = row["design"]["design_options"]
    results = row["sim_results"]
    record = {
        "design_id": canonical_design_id("GeneralizedCapNInterdigital", options),
        "source_id": row["notes"]["source_id"],
        "finger_count": int(options["finger_count"]),
        "C_NS_fF": results["north_to_south"],
        "C_NG_fF": results["north_to_ground"],
        "C_SG_fF": results["south_to_ground"],
        "C_NN_fF": results["north_to_north"],
        "C_SS_fF": results["south_to_south"],
        "C_GG_fF": results["ground_to_ground"],
    }
    record.update({
        key: parsed
        for key, value in options.items()
        if (parsed := numeric_option(value)) is not None
    })
    simulation_records.append(record)

full_simulations = pd.DataFrame(simulation_records)
available_counts = full_simulations["finger_count"].value_counts().sort_index()
expected_domains = pd.Index(range(2, 15), name="finger_count")
missing_domains = expected_domains.difference(available_counts.index)
undersized_domains = available_counts.reindex(expected_domains, fill_value=0).loc[
    lambda counts: counts < BALANCED_ROWS_PER_DOMAIN
]
if len(missing_domains) or len(undersized_domains):
    raise RuntimeError(
        "Tutorial 16b needs at least 1,260 rows in every finger-count domain; "
        f"missing={missing_domains.tolist()}, undersized={undersized_domains.to_dict()}."
    )

balanced_parts = []
for finger_count in expected_domains:
    domain = full_simulations.loc[
        full_simulations["finger_count"] == finger_count
    ].sort_values("design_id")
    if len(domain) > BALANCED_ROWS_PER_DOMAIN:
        domain = domain.sample(
            n=BALANCED_ROWS_PER_DOMAIN,
            random_state=RANDOM_SEED + int(finger_count),
        ).sort_values("design_id")
    balanced_parts.append(domain)
simulations = pd.concat(balanced_parts, ignore_index=True)
balanced_design_ids = frozenset(simulations["design_id"])

option_columns = [
    column for column in simulations.columns
    if column not in {"design_id", "source_id", *TARGET_COLUMNS}
]
geometry_columns = [
    column for column in option_columns
    if simulations[column].nunique(dropna=False) > 1
]
count_by_design = simulations.set_index("design_id")["finger_count"].to_dict()

rng = np.random.default_rng(14)
random_projection = rng.normal(
    0.0, 1.0 / np.sqrt(64), size=(9227, 64)
).astype(np.float32)
metadata_records, compact_batches, sketch_batches = [], [], []
raw_sums = {count: np.zeros(9227) for count in range(2, 15)}
raw_counts = {count: 0 for count in range(2, 15)}

parquet = pq.ParquetFile(embedding_path)
for batch in parquet.iter_batches(
    batch_size=64,
    columns=["design_id", "layout_id", "component_name", "embedding"],
):
    names = batch.column("component_name").to_pylist()
    design_ids = batch.column("design_id").to_pylist()
    keep = [
        index
        for index, (name, design_id) in enumerate(zip(names, design_ids))
        if name == "GeneralizedCapNInterdigital"
        and design_id in balanced_design_ids
    ]
    if not keep:
        continue
    matrix = np.asarray(
        [batch.column("embedding")[index].as_py() for index in keep],
        dtype=np.float32,
    )
    compact_batches.append(compress_v0_embeddings(matrix, pooled_shape_size=12).astype(np.float32))
    sketch_batches.append(matrix @ random_projection)
    for local_index, batch_index in enumerate(keep):
        design_id = design_ids[batch_index]
        finger_count = count_by_design[design_id]
        raw_sums[finger_count] += matrix[local_index]
        raw_counts[finger_count] += 1
        metadata_records.append({
            "design_id": design_id,
            "layout_id": batch.column("layout_id")[batch_index].as_py(),
        })

embedding_metadata = pd.DataFrame(metadata_records)
compact_v0 = np.vstack(compact_batches)
sketch = np.vstack(sketch_batches)
sketch -= sketch.mean(axis=0, keepdims=True)
u, singular_values, _ = np.linalg.svd(sketch, full_matrices=False)
projection = u[:, :10] * singular_values[:10]

data = embedding_metadata.merge(simulations, on="design_id", validate="one_to_one", sort=False)
data["projection_x"] = projection[:, 0]
data["projection_y"] = projection[:, 1]
raw_means = {
    count: (
        raw_sums[count] / raw_counts[count]
        if raw_counts[count]
        else np.zeros_like(raw_sums[count])
    )
    for count in raw_sums
}

overview = pd.DataFrame({
    "quantity": [
        "balanced simulation rows", "rows per finger-count domain",
        "unique design IDs", "unique immutable layouts",
        "varying geometry parameters", "capacitance targets",
    ],
    "value": [
        len(data), BALANCED_ROWS_PER_DOMAIN,
        data["design_id"].nunique(), data["layout_id"].nunique(),
        len(geometry_columns), len(TARGET_COLUMNS),
    ],
})
balanced_counts = data["finger_count"].value_counts().sort_index()
if len(data) != EXPECTED_BALANCED_ROWS or not balanced_counts.eq(
    BALANCED_ROWS_PER_DOMAIN
).all():
    raise RuntimeError(
        "Tutorial 16b balancing failed: expected 1,260 rows in each domain, "
        f"observed {balanced_counts.to_dict()} (total={len(data):,})."
    )
overview

,quantity,value
0,balanced simulation rows,16380
1,rows per finger-count domain,1260
2,unique design IDs,16380
3,unique immutable layouts,16380
4,varying geometry parameters,17
5,capacitance targets,6


In [3]:
# %% hide input
color_columns = geometry_columns + TARGET_COLUMNS
color_scales = {
    **{column: "Turbo" for column in geometry_columns},
    **{column: "Viridis" for column in TARGET_COLUMNS},
}
initial_column = "finger_count"
customdata = np.column_stack([
    data["finger_count"],
    data["finger_length"],
    data["finger_width"],
    data["C_NS_fF"],
])
figure = go.Figure(go.Scattergl(
    x=data["projection_x"],
    y=data["projection_y"],
    mode="markers",
    marker={
        "color": data[initial_column],
        "colorscale": color_scales[initial_column],
        "size": 5,
        "opacity": 0.62,
        "colorbar": {
            "title": initial_column.replace("_", " "),
            "x": 1.02,
            "xanchor": "left",
            "len": 0.78,
        },
    },
    customdata=customdata,
    hovertemplate=(
        "fingers=%{customdata[0]:.0f}<br>"
        "finger length=%{customdata[1]:.2f} um<br>"
        "finger width=%{customdata[2]:.2f} um<br>"
        "C(N,S)=%{customdata[3]:.3f} fF<extra></extra>"
    ),
))
buttons = [
    {
        "label": column.replace("_", " "),
        "method": "restyle",
        "args": [{
            "marker.color": [data[column].astype(np.float32)],
            "marker.colorscale": [color_scales[column]],
            "marker.colorbar.title": [column.replace("_", " ")],
            "marker.cmin": [float(data[column].min())],
            "marker.cmax": [float(data[column].max())],
        }],
    }
    for column in color_columns
]
figure.update_layout(
    title=f"All {len(data):,} generalized-NCap static-v0 embeddings",
    xaxis_title="randomized PCA sketch, axis 1",
    yaxis_title="randomized PCA sketch, axis 2",
    template="plotly_white",
    height=670,
    margin={"t": 120, "r": 150},
    updatemenus=[{
        "buttons": buttons, "direction": "down",
        "x": 0.01, "y": 1.12, "xanchor": "left", "yanchor": "top",
    }],
    annotations=[{
        "text": "Color every layout by geometry or capacitance:",
        "x": 0.01, "y": 1.18, "xref": "paper", "yref": "paper",
        "showarrow": False, "xanchor": "left",
    }],
)
figure.show()

In [4]:
# %% hide input
correlations = data[geometry_columns + TARGET_COLUMNS].corr(method="spearman").loc[
    geometry_columns, TARGET_COLUMNS
]
dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Coverage by finger count", "Six capacitance distributions",
        "C(N,S) across finger count", "Geometry-capacitance Spearman correlation",
    ],
    specs=[[{"type": "bar"}, {"type": "histogram"}],
           [{"type": "box"}, {"type": "heatmap"}]],
    vertical_spacing=0.16,
)
counts = data["finger_count"].value_counts().sort_index()
dashboard.add_trace(go.Bar(
    x=counts.index, y=counts.values, marker_color="#00798C",
    text=counts.values, textposition="outside", showlegend=False,
), row=1, col=1)
cap_colors = ["#D1495B", "#00798C", "#E9C46A", "#6A4C93", "#2A9D8F", "#F4A261"]
for column, color in zip(TARGET_COLUMNS, cap_colors):
    dashboard.add_trace(go.Histogram(
        x=data[column], name=column.replace("_", " "),
        opacity=0.45, marker_color=color, nbinsx=60,
    ), row=1, col=2)
dashboard.add_trace(go.Box(
    x=data["finger_count"], y=data["C_NS_fF"],
    marker_color="#D1495B", boxpoints=False, name="C(N,S)",
    showlegend=False,
), row=2, col=1)
dashboard.add_trace(go.Heatmap(
    z=correlations.to_numpy(), x=[c.replace("_", " ") for c in TARGET_COLUMNS],
    y=[c.replace("_", " ") for c in geometry_columns],
    colorscale="RdBu", zmid=0, zmin=-1, zmax=1,
    colorbar={
        "title": "Spearman ρ",
        "x": 1.01,
        "xanchor": "left",
        "y": 0.21,
        "len": 0.36,
    },
), row=2, col=2)
dashboard.update_xaxes(title_text="finger count", row=1, col=1)
dashboard.update_yaxes(title_text="designs", row=1, col=1)
dashboard.update_xaxes(title_text="capacitance (fF)", row=1, col=2)
dashboard.update_xaxes(title_text="finger count", row=2, col=1)
dashboard.update_yaxes(title_text="C(N,S) (fF)", row=2, col=1)
dashboard.update_layout(
    title="Geometry coverage and its capacitance response",
    barmode="overlay", template="plotly_white", height=900,
    margin={"t": 115, "r": 180, "b": 125},
    legend={
        "title": "capacitance histogram",
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": -0.10,
        "yanchor": "top",
    },
)
dashboard.show()

In [5]:
# %% hide input
parallel_sample = data.sample(min(5_000, len(data)), random_state=RANDOM_SEED)
dimensions = [
    {
        "label": column.replace("_", " "),
        "values": parallel_sample[column],
        "range": [data[column].min(), data[column].max()],
    }
    for column in geometry_columns
]
figure = go.Figure(go.Parcoords(
    line={
        "color": parallel_sample["C_NS_fF"],
        "colorscale": "Viridis",
        "showscale": True,
        "colorbar": {
            "title": "C(N,S) fF",
            "x": 1.03,
            "xanchor": "left",
            "len": 0.80,
        },
    },
    dimensions=dimensions,
))
figure.update_layout(
    title="Brush the swept geometry space and follow linked designs",
    height=650,
    margin={"l": 70, "r": 150, "t": 90, "b": 60},
)
figure.show()

### Recover average geometry motifs from embedding clusters

We cluster the first ten randomized-PCA coordinates, then reopen the v0 vectors and average their signed
96×96 shape blocks. These panels are **cluster prototypes**, not fabricated GDS files: bright regions are
consistently conductor-like, dark regions are consistently etch-like, and blurred edges reveal geometric
variation inside a cluster.

In [6]:
# %% hide input
def numpy_kmeans(matrix, clusters=9, seed=16, iterations=40):
    normalized = (matrix - matrix.mean(axis=0)) / np.maximum(matrix.std(axis=0), 1e-9)
    rng = np.random.default_rng(seed)
    centers = normalized[rng.choice(len(normalized), size=clusters, replace=False)]
    for _ in range(iterations):
        distances = np.sum((normalized[:, None, :] - centers[None, :, :]) ** 2, axis=2)
        labels = np.argmin(distances, axis=1)
        updated = np.vstack([
            normalized[labels == cluster].mean(axis=0)
            if np.any(labels == cluster) else centers[cluster]
            for cluster in range(clusters)
        ])
        if np.allclose(updated, centers, atol=1e-5):
            break
        centers = updated
    return labels


data["embedding_cluster"] = numpy_kmeans(projection[:, :10])
cluster_by_design = data.set_index("design_id")["embedding_cluster"].to_dict()
base_norm = float(np.linalg.norm(raw_means[BASE_FINGER_COUNT]))
if base_norm <= 1e-12:
    raise ValueError("The foundation embedding centroid has zero norm.")
base_centroid = raw_means[BASE_FINGER_COUNT] / base_norm
shape_sums = np.zeros((9, SHAPE_SIZE, SHAPE_SIZE), dtype=np.float64)
shape_counts = np.zeros(9, dtype=int)
similarity_by_design = {}

for batch in parquet.iter_batches(
    batch_size=64,
    columns=["design_id", "component_name", "embedding"],
):
    names = batch.column("component_name").to_pylist()
    for index, name in enumerate(names):
        if name != "GeneralizedCapNInterdigital":
            continue
        design_id = batch.column("design_id")[index].as_py()
        if design_id not in balanced_design_ids:
            continue
        vector = np.asarray(batch.column("embedding")[index].as_py(), dtype=np.float32)
        cluster = cluster_by_design[design_id]
        shape = vector[11:].reshape(SHAPE_SIZE, SHAPE_SIZE)
        shape_scale = float(np.max(np.abs(shape)))
        if shape_scale > 1e-12:
            shape = shape / shape_scale
        shape_sums[cluster] += shape
        shape_counts[cluster] += 1
        similarity_by_design[design_id] = float(vector @ base_centroid)

data["similarity_to_base"] = data["design_id"].map(similarity_by_design)
cluster_summary = data.groupby("embedding_cluster").agg(
    designs=("design_id", "size"),
    median_fingers=("finger_count", "median"),
    median_mutual_fF=("C_NS_fF", "median"),
)
cluster_figure = make_subplots(
    rows=3, cols=3,
    subplot_titles=[
        (
            f"cluster {cluster}<br>n={cluster_summary.loc[cluster, 'designs']:,}, "
            f"median fingers={cluster_summary.loc[cluster, 'median_fingers']:.0f}"
        )
        for cluster in range(9)
    ],
    horizontal_spacing=0.025, vertical_spacing=0.09,
)
for cluster in range(9):
    average_shape = np.divide(
        shape_sums[cluster],
        shape_counts[cluster],
        out=np.zeros_like(shape_sums[cluster]),
        where=shape_counts[cluster] > 0,
    )
    cluster_figure.add_trace(go.Heatmap(
        z=average_shape,
        colorscale=[[0, "#D1495B"], [0.5, "#F7F4EA"], [1, "#00798C"]],
        zmin=-1, zmax=1, showscale=False, hoverinfo="skip",
    ), row=cluster // 3 + 1, col=cluster % 3 + 1)
cluster_figure.update_xaxes(showticklabels=False)
cluster_figure.update_yaxes(showticklabels=False, autorange="reversed")
cluster_figure.update_layout(
    title="Nine average geometry motifs recovered from static-v0 clusters",
    height=900, template="plotly_white",
)
cluster_figure.show()

## 2. Choose a model that matches the scientific question

The capacitance map is nonlinear across finger count, but this experiment also needs a source model whose
knowledge can be transferred transparently. We therefore use:

- **compact v0**: the parameter sum, ten moments, and a 12×12 average-pooled shape tensor (155 values);
- **random Fourier features**: a deterministic approximation to an RBF kernel that restores smooth nonlinear
  interactions without fitting labels;
- **multi-output ridge**: one convex head predicts `C(N,S)`, `C(N,G)`, and `C(S,G)` together;
- **source-prior adaptation**: the target head learns a correction around the frozen foundation weights.

This is not claimed to be the universally most accurate regressor. It is the best controlled instrument for
this tutorial: fast, reproducible, nonlinear, multi-output, and explicit about what transfers.

The first run writes every fitted weight matrix and its metrics to a fingerprinted checkpoint. Later runs
restore those weights when the dataset and experiment configuration are unchanged.

In [7]:
projector = V0KernelFeatureProjector(
    pooled_shape_size=12,
    kernel_dimensions=128,
    random_seed=RANDOM_SEED,
)
kernel_features = projector.fit_transform_compact(compact_v0)
scalar_projector = V0KernelFeatureProjector(
    pooled_shape_size=12,
    kernel_dimensions=64,
    random_seed=RANDOM_SEED + 1,
)
scalar_features = scalar_projector.fit_transform_compact(compact_v0[:, :11])
shape_projector = V0KernelFeatureProjector(
    pooled_shape_size=12,
    kernel_dimensions=128,
    random_seed=RANDOM_SEED + 2,
)
shape_features = shape_projector.fit_transform_compact(compact_v0[:, 11:])
finger_counts = data["finger_count"].to_numpy(dtype=int)
count_values = np.arange(2, 15)
count_one_hot = (finger_counts[:, None] == count_values[None, :]).astype(float)
normalized_count = (finger_counts - finger_counts.mean()) / finger_counts.std()
domain_features = np.column_stack([
    count_one_hot,
    normalized_count,
    normalized_count**2,
    normalized_count**3,
])
model_features = np.column_stack([kernel_features, domain_features])
feature_sets = {
    "full embedding + domain": model_features,
    "embedding only": kernel_features,
    "scalar geometry + domain": np.column_stack([scalar_features, domain_features]),
    "shape geometry + domain": np.column_stack([shape_features, domain_features]),
    "domain identity only": domain_features,
}
targets = data[ML_TARGETS].to_numpy(dtype=float)


def partition_splits(repeat, test_fraction=0.25):
    splits = {}
    seed = RANDOM_SEED + 100_000 * repeat
    for count in sorted(np.unique(finger_counts)):
        indices = np.flatnonzero(finger_counts == count)
        order = np.random.default_rng(seed + 1_000 * count).permutation(indices)
        test_size = round(test_fraction * len(indices))
        splits[count] = {"pool": order[test_size:], "test": order[:test_size]}
    return splits


splits_by_repeat = {
    repeat: partition_splits(repeat)
    for repeat in range(N_REPEATS)
}
tuning_splits = partition_splits(10_000)
base_order = np.random.default_rng(RANDOM_SEED + 800).permutation(
    tuning_splits[BASE_FINGER_COUNT]["pool"]
)
validation_size = round(0.20 * len(base_order))
base_validation, base_training = base_order[:validation_size], base_order[validation_size:]
MODEL_CONFIG = {
    "checkpoint_version": 4,
    "base_finger_count": BASE_FINGER_COUNT,
    "fractions": FRACTIONS,
    "repeats": N_REPEATS,
    "split_strategy": "independent_stratified_repeated_holdout",
    "random_seed": RANDOM_SEED,
    "test_fraction": 0.25,
    "confidence_level": CONFIDENCE_LEVEL,
    "pooled_shape_size": projector.pooled_shape_size,
    "kernel_dimensions": projector.kernel_dimensions,
    "alpha_candidates": [0.1, 1.0, 10.0, 100.0],
    "targets": ML_TARGETS,
}
fingerprint = hashlib.sha256()
fingerprint.update(json.dumps(MODEL_CONFIG, sort_keys=True).encode())
fingerprint.update("\n".join(data["design_id"]).encode())
fingerprint.update(np.ascontiguousarray(targets, dtype="<f8").tobytes())
EXPERIMENT_ID = fingerprint.hexdigest()[:20]
CHECKPOINT_DIR = CACHE_ROOT / EXPERIMENT_ID
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = CHECKPOINT_DIR / "learning-curves.parquet"
WEIGHTS_PATH = CHECKPOINT_DIR / "trained-weights.npz"
INDEX_PATH = CHECKPOINT_DIR / "model-index.json"
METADATA_PATH = CHECKPOINT_DIR / "metadata.json"

pd.DataFrame({
    "setting": [
        "experiment fingerprint", "checkpoint directory", "model feature width",
        "independent holdouts", "confidence level",
    ],
    "value": [
        EXPERIMENT_ID, str(CHECKPOINT_DIR), model_features.shape[1],
        N_REPEATS, CONFIDENCE_LEVEL,
    ],
})


,setting,value
0,experiment fingerprint,18005b4aa44ee1a4581f
1,checkpoint directory,/Users/shanto/.cache/squadds/tutorial16b/18005...
2,model feature width,299
3,independent holdouts,12
4,confidence level,0.95


In [8]:
# %% hide input
architecture = go.Figure(go.Sankey(
    node={
        "label": [
            "v0 parameter sum (1)", "v0 moments (10)", "v0 shape (96×96)",
            "12×12 pooled shape", "RBF random features", "finger-count basis",
            "convex multi-output head", "3 capacitances (fF)",
        ],
        "color": [
            "#E9C46A", "#F4A261", "#00798C", "#2A9D8F",
            "#6A4C93", "#D1495B", "#264653", "#E76F51",
        ],
        "pad": 18, "thickness": 22,
    },
    link={
        "source": [0, 1, 2, 3, 4, 5, 6],
        "target": [4, 4, 3, 4, 6, 6, 7],
        "value": [1, 10, 144, 144, model_features.shape[1] - 16, 16, 3],
    },
))
architecture.update_layout(
    title="Checkpointed model pipeline (alpha selected once, then restored)",
    height=480, template="plotly_white",
)
architecture.show()

## 3. Specialists versus two pooled generalists

Each exact finger count is a domain. We run **12 independently seeded, finger-count-stratified holdouts**.
Every repeat reserves 25% of each domain as test data; the remaining 75% supplies nested prefixes at every
integer percentage from 1% through 100%.

At every percentage and repeat we fit exactly two generalists:

- **coverage-matched generalist**: sees that percentage from every finger-count pool, so it receives about
  13 times as many rows as one specialist;
- **budget-matched generalist**: receives the same total row budget as one typical specialist, distributed
  proportionally across all 13 finger-count pools.

Every comparison is paired within the same repeat, test rows, and label budget. Curves report the repeated-
holdout mean and 95% Student-t confidence interval. These intervals quantify robustness to data splitting;
they do not turn this finite simulated catalogue into independent physical experiments.


In this balanced companion, every domain begins with exactly 1,260 designs. The 25% held-out split therefore leaves the same 945-row training pool in every domain, and each percentage corresponds to the same absolute per-domain sample budget. Coverage-matched pooled generalists still receive 13 times as many rows as one specialist; budget-matched pooled generalists receive the same total row count as one specialist, proportionally sampled across all 13 domains.

In [9]:
def macro_metrics(expected, predicted):
    return regression_scores(expected, predicted, ML_TARGETS).query("target == 'macro'").iloc[0]


def ci95(values):
    clean = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(clean) < 2:
        return np.nan
    return float(
        student_t.ppf(0.5 + CONFIDENCE_LEVEL / 2, len(clean) - 1)
        * clean.std(ddof=1) / np.sqrt(len(clean))
    )


def bh_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(values), np.nan)
    valid = np.flatnonzero(np.isfinite(values))
    if not len(valid):
        return adjusted
    order = valid[np.argsort(values[valid])]
    ranked = values[order] * len(order) / np.arange(1, len(order) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


def nested_training_indices(repeat):
    return {
        count: np.random.default_rng(
            RANDOM_SEED + repeat * 10_000 + count * 100
        ).permutation(split["pool"])
        for count, split in splits_by_repeat[repeat].items()
    }


def proportional_generalist_indices(orders, budget):
    counts = np.asarray(sorted(orders))
    available = np.asarray([len(orders[count]) for count in counts])
    exact = budget * available / available.sum()
    allocation = np.floor(exact).astype(int)
    remainder_order = np.argsort(-(exact - allocation), kind="stable")
    allocation[remainder_order[:budget - allocation.sum()]] += 1
    return np.concatenate([
        orders[count][:rows]
        for count, rows in zip(counts, allocation)
    ])


def restore_model(model_key):
    row = model_index.loc[model_index["model_key"] == model_key].iloc[0]
    model = TransferRidgeRegressor(float(row["alpha"]))
    model.weights_ = trained_weights[int(row["weight_index"])].copy()
    return model


checkpoint_complete = all(
    path.exists()
    for path in [METRICS_PATH, WEIGHTS_PATH, INDEX_PATH, METADATA_PATH]
)
if checkpoint_complete:
    metadata = json.loads(METADATA_PATH.read_text())
    curves = pd.read_parquet(METRICS_PATH)
    trained_weights = np.load(WEIGHTS_PATH)["weights"]
    model_index = pd.DataFrame(json.loads(INDEX_PATH.read_text()))
    alpha_results = pd.DataFrame(metadata["alpha_results"])
    ALPHA = float(metadata["selected_alpha"])
    foundation_models = [
        restore_model(f"foundation-r{repeat}")
        for repeat in range(MODEL_CONFIG["repeats"])
    ]
    checkpoint_state = f"Loaded {len(model_index):,} trained models from checkpoint."
else:
    model_weights = []
    model_records = []

    def save_model(model_key, model, **fields):
        model_records.append({
            "model_key": model_key,
            "weight_index": len(model_weights),
            "alpha": model.alpha,
            **fields,
        })
        model_weights.append(model.weights_.copy())

    alpha_records = []
    for alpha in MODEL_CONFIG["alpha_candidates"]:
        candidate = TransferRidgeRegressor(alpha).fit(
            model_features[base_training], targets[base_training]
        )
        score = macro_metrics(
            targets[base_validation],
            candidate.predict(model_features[base_validation]),
        )
        alpha_records.append({"alpha": alpha, **score.to_dict()})
        save_model(f"alpha-candidate-{alpha:g}", candidate, method="alpha_candidate")
    alpha_results = pd.DataFrame(alpha_records)
    ALPHA = float(alpha_results.loc[alpha_results["r2"].idxmax(), "alpha"])

    curve_records = []
    foundation_models = []
    for repeat in range(MODEL_CONFIG["repeats"]):
        repeat_splits = splits_by_repeat[repeat]
        orders = nested_training_indices(repeat)
        foundation = TransferRidgeRegressor(ALPHA).fit(
            model_features[repeat_splits[BASE_FINGER_COUNT]["pool"]],
            targets[repeat_splits[BASE_FINGER_COUNT]["pool"]],
        )
        foundation_models.append(foundation)
        save_model(
            f"foundation-r{repeat}", foundation,
            method="foundation", repeat=repeat,
        )
        for fraction in FRACTIONS:
            fraction_key = f"{fraction:.2f}"
            selected = {
                count: order[:max(1, round(fraction * len(order)))]
                for count, order in orders.items()
            }
            coverage_pool = np.concatenate(list(selected.values()))
            specialist_budget = round(np.mean([
                len(indices) for indices in selected.values()
            ]))
            budget_pool = proportional_generalist_indices(orders, specialist_budget)
            coverage_generalist = TransferRidgeRegressor(ALPHA).fit(
                model_features[coverage_pool], targets[coverage_pool]
            )
            save_model(
                f"generalist-coverage-r{repeat}-f{fraction_key}",
                coverage_generalist, method="generalist_coverage",
                repeat=repeat, fraction=fraction, training_rows=len(coverage_pool),
            )
            budget_generalist = TransferRidgeRegressor(ALPHA).fit(
                model_features[budget_pool], targets[budget_pool]
            )
            save_model(
                f"generalist-budget-r{repeat}-f{fraction_key}",
                budget_generalist, method="generalist_budget",
                repeat=repeat, fraction=fraction, training_rows=len(budget_pool),
            )
            for count, split in repeat_splits.items():
                test = split["test"]
                specialist = TransferRidgeRegressor(ALPHA).fit(
                    model_features[selected[count]], targets[selected[count]]
                )
                save_model(
                    f"specialist-c{count}-r{repeat}-f{fraction_key}",
                    specialist, method="specialist", finger_count=int(count),
                    repeat=repeat, fraction=fraction,
                )
                models = {
                    "specialist": (specialist, len(selected[count])),
                    "generalist_coverage": (coverage_generalist, len(coverage_pool)),
                    "generalist_budget": (budget_generalist, len(budget_pool)),
                }
                if count != BASE_FINGER_COUNT:
                    transfer = TransferRidgeRegressor(ALPHA).fit(
                        model_features[selected[count]], targets[selected[count]],
                        prior=foundation,
                    )
                    save_model(
                        f"transfer-c{count}-r{repeat}-f{fraction_key}",
                        transfer, method="transfer", finger_count=int(count),
                        repeat=repeat, fraction=fraction,
                    )
                    models["transfer"] = (transfer, len(selected[count]))
                for method, (model, training_rows) in models.items():
                    score = macro_metrics(
                        targets[test], model.predict(model_features[test])
                    )
                    curve_records.append({
                        "finger_count": count, "fraction": fraction,
                        "labels": len(selected[count]), "training_rows": training_rows,
                        "repeat": repeat, "method": method, **score.to_dict(),
                    })

    curves = pd.DataFrame(curve_records)
    trained_weights = np.stack(model_weights)
    model_index = pd.DataFrame(model_records)
    curves.to_parquet(METRICS_PATH, index=False)
    with WEIGHTS_PATH.open("wb") as checkpoint_file:
        np.savez_compressed(checkpoint_file, weights=trained_weights)
    INDEX_PATH.write_text(
        json.dumps(json.loads(model_index.to_json(orient="records")), indent=2)
    )
    METADATA_PATH.write_text(json.dumps({
        "experiment_id": EXPERIMENT_ID,
        "selected_alpha": ALPHA,
        "model_config": MODEL_CONFIG,
        "alpha_results": json.loads(alpha_results.to_json(orient="records")),
    }, indent=2))
    checkpoint_state = f"Trained and checkpointed {len(model_index):,} models."

foundation = foundation_models[0]
curve_summary = curves.groupby(
    ["finger_count", "fraction", "method"], as_index=False
).agg(
    r2=("r2", "mean"), r2_ci95=("r2", ci95),
    mae_fF=("mae", "mean"), within_5_percent=("within_5_percent", "mean"),
    labels=("labels", "mean"), training_rows=("training_rows", "mean"),
)
curve_summary["r2_lower"] = curve_summary["r2"] - curve_summary["r2_ci95"]
curve_summary["r2_upper"] = curve_summary["r2"] + curve_summary["r2_ci95"]
print(checkpoint_state)
print(f"Selected alpha: {ALPHA:g}")
print(f"Independent stratified holdouts: {MODEL_CONFIG['repeats']}")
print(f"Checkpoint: {CHECKPOINT_DIR}")
curve_summary.query(
    "fraction in [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 1.0]"
).head(12).round(4)


Loaded 32,416 trained models from checkpoint.
Selected alpha: 0.1
Independent stratified holdouts: 12
Checkpoint: /Users/shanto/.cache/squadds/tutorial16b/18005b4aa44ee1a4581f


,finger_count,fraction,method,r2,r2_ci95,mae_fF,within_5_percent,labels,training_rows,r2_lower,r2_upper
0,2,0.01,generalist_budget,-1.9626,1.1221,0.9504,11.4198,9.0,9.0,-3.0847,-0.8406
1,2,0.01,generalist_coverage,-5.8202,1.5019,1.2815,10.2822,9.0,117.0,-7.3221,-4.3183
2,2,0.01,specialist,0.3847,0.2135,0.4479,26.6402,9.0,9.0,0.1712,0.5982
3,2,0.01,transfer,-3.9305,1.4693,0.9789,14.1534,9.0,9.0,-5.3998,-2.4612
16,2,0.05,generalist_budget,-16.2444,7.5850,1.9381,7.4515,47.0,47.0,-23.8294,-8.6595
17,2,0.05,generalist_coverage,0.0930,0.2898,0.4373,28.2099,47.0,611.0,-0.1968,0.3828
18,2,0.05,specialist,0.9596,0.0148,0.0885,76.0847,47.0,47.0,0.9448,0.9744
19,2,0.05,transfer,0.7187,0.1176,0.1849,56.2522,47.0,47.0,0.6011,0.8363
36,2,0.10,generalist_budget,-7.7402,2.8314,1.4610,9.1270,94.0,94.0,-10.5717,-4.9088
37,2,0.10,generalist_coverage,0.4312,0.1170,0.3418,34.1093,94.0,1222.0,0.3142,0.5483


In [10]:
# %% hide input
colors = {
    count: f"hsl({(count - 2) * 300 / 12:.0f},65%,42%)"
    for count in range(2, 15)
}
specialist_figure = go.Figure()
for count in range(2, 15):
    frame = curve_summary.query(
        "finger_count == @count and method == 'specialist'"
    ).sort_values("fraction")
    specialist_figure.add_trace(go.Scatter(
        x=100 * frame["fraction"], y=frame["r2"],
        mode="lines+markers", name=f"{count} fingers",
        error_y={"type": "data", "array": frame["r2_ci95"], "visible": True},
        line={"color": colors[count], "width": 2.5},
        marker={"size": 7},
        customdata=frame["labels"].round().astype(int),
        hovertemplate=(
            f"<b>{count} fingers</b><br>pool=%{{x:.0f}}%"
            "<br>labels=%{customdata}<br>macro R²=%{y:.4f}<extra></extra>"
        ),
    ))
specialist_figure.update_layout(
    title="Thirteen specialist learning curves (mean ± 95% CI over 12 holdouts)",
    xaxis_title="fraction of each domain's training pool (%)",
    yaxis_title="held-out macro R²",
    yaxis_range=[0, 1.01],
    template="plotly_white", height=650,
    margin={"b": 145},
    legend={
        "title": "specialist",
        "orientation": "h",
        "y": -0.08,
        "yanchor": "top",
    },
)
specialist_figure.show()

In [11]:
# %% hide input
comparison_figure = go.Figure()
for count in range(2, 15):
    frame = curve_summary.query(
        "finger_count == @count and method == 'specialist'"
    ).sort_values("fraction")
    comparison_figure.add_trace(go.Scatter(
        x=100 * frame["fraction"], y=frame["r2"],
        mode="lines+markers",
        name=(
            "13 finger-count specialists (dotted)"
            if count == 2
            else f"specialist · {count} fingers"
        ),
        legendgroup="specialists",
        showlegend=count == 2,
        line={"color": colors[count], "dash": "dot", "width": 1.8},
        marker={"size": 5},
        opacity=0.55,
        hovertemplate=(
            f"<b>{count}-finger specialist</b><br>"
            "pool=%{x:.0f}%<br>held-out macro R²=%{y:.4f}<extra></extra>"
        ),
    ))

generalist_by_repeat = curves.query(
    "method in ['generalist_coverage', 'generalist_budget']"
).groupby(["fraction", "method", "repeat"], as_index=False).agg(
    r2=("r2", "mean"),
    training_rows=("training_rows", "first"),
)
generalist_plot = generalist_by_repeat.groupby(
    ["fraction", "method"], as_index=False
).agg(
    r2=("r2", "mean"), r2_ci95=("r2", ci95),
    training_rows=("training_rows", "mean"),
)
for method, label, color, dash in [
    (
        "generalist_coverage",
        "coverage-matched generalist (~13× labels)",
        "#111827",
        "solid",
    ),
    (
        "generalist_budget",
        "budget-matched generalist (1× labels)",
        "#00798C",
        "dash",
    ),
]:
    frame = generalist_plot.query("method == @method").sort_values("fraction")
    comparison_figure.add_trace(go.Scatter(
        x=100 * frame["fraction"], y=frame["r2"],
        mode="lines+markers",
        error_y={"type": "data", "array": frame["r2_ci95"], "visible": True},
        name=label,
        customdata=frame["training_rows"].round().astype(int),
        line={"color": color, "dash": dash, "width": 4},
        marker={"size": 9},
        hovertemplate=(
            f"<b>{label}</b><br>pool=%{{x:.0f}}%"
            "<br>total training rows=%{customdata:,}"
            "<br>domain-macro R²=%{y:.4f}<extra></extra>"
        ),
    ))
comparison_figure.update_layout(
    title="Two generalists versus specialists (mean ± 95% CI over 12 holdouts)",
    xaxis_title="fraction of each domain's training pool (%)",
    yaxis_title="held-out domain-macro R²",
    yaxis_range=[0, 1.01],
    template="plotly_white", height=680,
    margin={"b": 120},
    legend={
        "title": "model",
        "orientation": "h",
        "y": -0.08,
        "yanchor": "top",
    },
)
comparison_figure.show()

### 3.1 Ablation: which geometry information earns the accuracy?

An ablation is a controlled removal experiment. Using the same 12 holdouts and exactly the same selected
training rows, we compare the full representation with four removals: no explicit domain identity, scalar
geometry only, pooled shape geometry only, and domain identity without an embedding. Seven label budgets keep
this diagnostic readable. Error bars are paired 95% confidence intervals; adjusted `q` values compare each
ablation with the full model.


In [12]:
ABLATION_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 1.00]
ABLATION_DIR = CHECKPOINT_DIR / "feature-ablation-v1"
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_METRICS_PATH = ABLATION_DIR / "metrics.parquet"
ABLATION_WEIGHTS_PATH = ABLATION_DIR / "trained-weights.npz"
if ABLATION_METRICS_PATH.exists() and ABLATION_WEIGHTS_PATH.exists():
    ablation_curves = pd.read_parquet(ABLATION_METRICS_PATH)
    ablation_state = "Loaded feature-ablation models from checkpoint."
else:
    ablation_records = []
    ablation_weights = {}
    for repeat in range(N_REPEATS):
        orders = nested_training_indices(repeat)
        repeat_splits = splits_by_repeat[repeat]
        for fraction in ABLATION_FRACTIONS:
            selected = np.concatenate([
                order[:max(1, round(fraction * len(order)))]
                for order in orders.values()
            ])
            for feature_name, features in feature_sets.items():
                model = TransferRidgeRegressor(ALPHA).fit(
                    features[selected], targets[selected]
                )
                key = (
                    feature_name.lower().replace(" ", "_").replace("+", "plus")
                    + f"-r{repeat}-f{fraction:.2f}"
                )
                ablation_weights[key] = model.weights_.copy()
                domain_scores = [
                    macro_metrics(
                        targets[split["test"]],
                        model.predict(features[split["test"]]),
                    )["r2"]
                    for split in repeat_splits.values()
                ]
                ablation_records.append({
                    "feature_set": feature_name,
                    "repeat": repeat,
                    "fraction": fraction,
                    "r2": float(np.mean(domain_scores)),
                    "training_rows": len(selected),
                })
    ablation_curves = pd.DataFrame(ablation_records)
    ablation_curves.to_parquet(ABLATION_METRICS_PATH, index=False)
    np.savez_compressed(ABLATION_WEIGHTS_PATH, **ablation_weights)
    ablation_state = f"Trained and checkpointed {len(ablation_weights):,} ablation models."

ablation_summary = ablation_curves.groupby(
    ["feature_set", "fraction"], as_index=False
).agg(r2=("r2", "mean"), r2_ci95=("r2", ci95), training_rows=("training_rows", "mean"))
full_ablation = ablation_curves.query(
    "feature_set == 'full embedding + domain'"
)[["repeat", "fraction", "r2"]].rename(columns={"r2": "r2_full"})
ablation_pairs = ablation_curves.merge(full_ablation, on=["repeat", "fraction"])
ablation_pairs["delta_vs_full"] = ablation_pairs["r2"] - ablation_pairs["r2_full"]
ablation_tests = ablation_pairs.query(
    "feature_set != 'full embedding + domain'"
).groupby(["feature_set", "fraction"])["delta_vs_full"].apply(
    lambda values: pd.Series({
        "delta_vs_full": values.mean(),
        "delta_ci95": ci95(values),
        "p_value": ttest_1samp(values, 0.0).pvalue,
    })
).unstack().reset_index()
ablation_tests["q_value"] = bh_adjust(ablation_tests["p_value"])
print(ablation_state)

ablation_figure = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Representation learning curves",
        "Paired change from full model at 10% labels",
    ],
    horizontal_spacing=0.14,
)
ablation_colors = {
    "full embedding + domain": "#111827",
    "embedding only": "#00798C",
    "scalar geometry + domain": "#F4A261",
    "shape geometry + domain": "#6A4C93",
    "domain identity only": "#D1495B",
}
for feature_name, color in ablation_colors.items():
    frame = ablation_summary.query("feature_set == @feature_name").sort_values("fraction")
    ablation_figure.add_trace(go.Scatter(
        x=100 * frame["fraction"], y=frame["r2"],
        error_y={"type": "data", "array": frame["r2_ci95"], "visible": True},
        mode="lines+markers", name=feature_name,
        line={"color": color, "width": 3}, marker={"size": 7},
        hovertemplate=(
            f"<b>{feature_name}</b><br>pool=%{{x:.0f}}%"
            "<br>domain-macro R²=%{y:.4f}<extra></extra>"
        ),
    ), row=1, col=1)
ten_percent = ablation_tests.query("fraction == 0.10").sort_values("delta_vs_full")
ablation_figure.add_trace(go.Bar(
    x=ten_percent["feature_set"], y=ten_percent["delta_vs_full"],
    error_y={"type": "data", "array": ten_percent["delta_ci95"], "visible": True},
    marker_color=[ablation_colors[name] for name in ten_percent["feature_set"]],
    customdata=ten_percent["q_value"],
    showlegend=False,
    hovertemplate=(
        "%{x}<br>paired ΔR²=%{y:.4f}"
        "<br>BH q=%{customdata:.3g}<extra></extra>"
    ),
), row=1, col=2)
ablation_figure.add_hline(y=0, line_dash="dash", line_color="#6B7280", row=1, col=2)
ablation_figure.update_xaxes(title_text="training pool used (%)", row=1, col=1)
ablation_figure.update_yaxes(title_text="held-out domain-macro R²", row=1, col=1)
ablation_figure.update_xaxes(title_text="feature ablation", tickangle=-25, row=1, col=2)
ablation_figure.update_yaxes(title_text="paired ΔR² versus full", row=1, col=2)
ablation_figure.update_layout(
    title="Feature ablation over 12 independent holdouts",
    template="plotly_white", height=680, margin={"b": 190},
    legend={"orientation": "h", "x": 0.5, "xanchor": "center", "y": -0.19},
)
ablation_figure.show()


Loaded feature-ablation models from checkpoint.


## 4. Transfer from a count-8 foundation

Finger count 8 is tied for the largest domain and lies near the center of the 2–14 range. Each of the 12
holdouts fits its own count-8 foundation, target specialist, and adapted transfer head. Comparisons therefore
remain paired on split, labels, and test rows.

We report mean held-out macro R² and MAE, **paired transfer gain**, 95% confidence intervals, and
Benjamini–Hochberg-adjusted p-values. A cell is called robust only when its paired interval excludes zero and
its false-discovery-rate-adjusted `q < 0.05`.


In [13]:
similarity_records = []
for count in range(2, 15):
    if count != BASE_FINGER_COUNT:
        similarity_records.append({
            "finger_count": count,
            "mean_pairwise_cosine": float(raw_means[BASE_FINGER_COUNT] @ raw_means[count]),
        })
domain_similarity = pd.DataFrame(similarity_records)

paired_transfer = curves.query(
    "method in ['specialist', 'transfer']"
).pivot(
    index=["finger_count", "fraction", "repeat"],
    columns="method",
    values=["r2", "mae", "within_5_percent"],
).reset_index()
paired_transfer.columns = [
    "_".join(str(part) for part in column if part).rstrip("_")
    for column in paired_transfer.columns
]
paired_transfer["r2_gain"] = (
    paired_transfer["r2_transfer"] - paired_transfer["r2_specialist"]
)
paired_transfer["mae_reduction_fF"] = (
    paired_transfer["mae_specialist"] - paired_transfer["mae_transfer"]
)
transfer_summary = paired_transfer.groupby(
    ["finger_count", "fraction"], as_index=False
).agg(
    r2_specialist=("r2_specialist", "mean"),
    r2_transfer=("r2_transfer", "mean"),
    r2_gain=("r2_gain", "mean"),
    r2_gain_ci95=("r2_gain", ci95),
    mae_specialist=("mae_specialist", "mean"),
    mae_transfer=("mae_transfer", "mean"),
    mae_reduction_fF=("mae_reduction_fF", "mean"),
    within_5_percent_specialist=("within_5_percent_specialist", "mean"),
    within_5_percent_transfer=("within_5_percent_transfer", "mean"),
)
gain_tests = paired_transfer.groupby(
    ["finger_count", "fraction"]
)["r2_gain"].apply(
    lambda values: ttest_1samp(values, 0.0).pvalue
).reset_index(name="p_value")
gain_tests["q_value"] = bh_adjust(gain_tests["p_value"])
transfer_summary = transfer_summary.merge(
    gain_tests, on=["finger_count", "fraction"]
).merge(domain_similarity, on="finger_count")
transfer_summary["r2_gain_lower"] = (
    transfer_summary["r2_gain"] - transfer_summary["r2_gain_ci95"]
)
transfer_summary["r2_gain_upper"] = (
    transfer_summary["r2_gain"] + transfer_summary["r2_gain_ci95"]
)
transfer_summary["robust_gain"] = (
    (transfer_summary["q_value"] < 0.05)
    & (
        (transfer_summary["r2_gain_lower"] > 0)
        | (transfer_summary["r2_gain_upper"] < 0)
    )
)
full_specialist = curve_summary.query(
    "method == 'specialist' and fraction == 1.0"
)[["finger_count", "r2"]].rename(columns={"r2": "full_specialist_r2"})
transfer_summary = transfer_summary.merge(full_specialist, on="finger_count")
transfer_summary["fraction_of_full"] = (
    transfer_summary["r2_transfer"] / transfer_summary["full_specialist_r2"]
)
transfer_summary.head().round(4)


,finger_count,fraction,r2_specialist,r2_transfer,r2_gain,r2_gain_ci95,mae_specialist,mae_transfer,mae_reduction_fF,within_5_percent_specialist,within_5_percent_transfer,p_value,q_value,mean_pairwise_cosine,r2_gain_lower,r2_gain_upper,robust_gain,full_specialist_r2,fraction_of_full
0,2,0.01,0.3847,-3.9305,-4.3152,1.4387,0.4479,0.9789,-0.5310,26.6402,14.1534,0.0000,0.0001,0.1278,-5.7539,-2.8764,True,0.9915,-3.9642
1,2,0.02,0.8799,-0.2801,-1.1600,0.4401,0.1527,0.4174,-0.2648,61.3933,32.2840,0.0001,0.0003,0.1278,-1.6001,-0.7199,True,0.9915,-0.2825
2,2,0.03,0.9261,0.1166,-0.8095,0.4204,0.1168,0.3130,-0.1962,69.7002,42.2751,0.0014,0.0029,0.1278,-1.2299,-0.3891,True,0.9915,0.1176
3,2,0.04,0.9544,0.4568,-0.4976,0.3538,0.0924,0.2378,-0.1454,75.4762,50.6614,0.0102,0.0185,0.1278,-0.8514,-0.1438,True,0.9915,0.4607
4,2,0.05,0.9596,0.7187,-0.2409,0.1096,0.0885,0.1849,-0.0965,76.0847,56.2522,0.0005,0.0012,0.1278,-0.3505,-0.1313,True,0.9915,0.7248


In [14]:
# %% hide input
gain_matrix = transfer_summary.pivot(
    index="finger_count", columns="fraction", values="r2_gain"
).sort_index()
lower_matrix = transfer_summary.pivot(
    index="finger_count", columns="fraction", values="r2_gain_lower"
).reindex_like(gain_matrix)
upper_matrix = transfer_summary.pivot(
    index="finger_count", columns="fraction", values="r2_gain_upper"
).reindex_like(gain_matrix)
q_matrix = transfer_summary.pivot(
    index="finger_count", columns="fraction", values="q_value"
).reindex_like(gain_matrix)
robust_matrix = transfer_summary.pivot(
    index="finger_count", columns="fraction", values="robust_gain"
).reindex_like(gain_matrix)
figure = go.Figure(go.Heatmap(
    z=gain_matrix.to_numpy(),
    x=100 * gain_matrix.columns.to_numpy(),
    y=gain_matrix.index,
    colorscale="RdBu", zmid=0,
    colorbar={"title": "mean Δ macro R²", "x": 1.02, "xanchor": "left"},
    text=np.where(robust_matrix.to_numpy(), "•", ""),
    texttemplate="%{text}",
    customdata=np.stack([
        lower_matrix.to_numpy(), upper_matrix.to_numpy(), q_matrix.to_numpy()
    ], axis=-1),
    hovertemplate=(
        "target fingers=%{y}<br>target pool=%{x:.0f}%"
        "<br>mean transfer gain=%{z:.4f}"
        "<br>95% CI=[%{customdata[0]:.4f}, %{customdata[1]:.4f}]"
        "<br>BH q=%{customdata[2]:.3g}<extra></extra>"
    ),
))
figure.update_layout(
    title="Paired transfer gain across 12 holdouts (• = 95% CI excludes zero and BH q < 0.05)",
    xaxis_title="fraction of target training pool (%)",
    yaxis_title="target finger count",
    template="plotly_white", height=640, margin={"r": 155},
)
figure.show()


In [15]:
# %% hide input
similarity_lines = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Each target's gain as labels increase",
        "At a fixed budget, gain follows similarity",
    ],
    horizontal_spacing=0.13,
)
similarity_min = transfer_summary["mean_pairwise_cosine"].min()
similarity_max = transfer_summary["mean_pairwise_cosine"].max()
for count in sorted(transfer_summary["finger_count"].unique()):
    frame = transfer_summary.query("finger_count == @count").sort_values("fraction")
    cosine = float(frame["mean_pairwise_cosine"].iloc[0])
    color_position = (cosine - similarity_min) / max(
        similarity_max - similarity_min, 1e-12
    )
    color = sample_colorscale("Viridis", [color_position])[0]
    similarity_lines.add_trace(go.Scatter(
        x=100 * frame["fraction"],
        y=frame["r2_gain"],
        mode="lines+markers",
        name=f"{count} fingers · cosine={cosine:.3f}",
        line={"color": color, "width": 2.5},
        marker={"size": 7},
        legendgroup=f"target-{count}",
        hovertemplate=(
            f"<b>target: {count} fingers</b><br>"
            f"mean pairwise cosine={cosine:.4f}<br>"
            "target pool=%{x:.0f}%<br>transfer ΔR²=%{y:.4f}<extra></extra>"
        ),
    ), row=1, col=1)

for fraction, color in zip(
    [0.05, 0.10, 0.25, 0.50, 1.00],
    ["#D1495B", "#F4A261", "#E9C46A", "#00798C", "#264653"],
):
    frame = transfer_summary.query("fraction == @fraction").sort_values(
        "mean_pairwise_cosine"
    )
    similarity_lines.add_trace(go.Scatter(
        x=frame["mean_pairwise_cosine"],
        y=frame["r2_gain"],
        mode="lines+markers",
        name=f"{100 * fraction:.0f}% target pool",
        line={"color": color, "width": 3},
        marker={"size": 7},
        legendgroup=f"fraction-{fraction}",
        hovertemplate=(
            f"<b>{100 * fraction:.0f}% target pool</b><br>"
            "mean pairwise cosine=%{x:.4f}<br>"
            "transfer ΔR²=%{y:.4f}<extra></extra>"
        ),
    ), row=1, col=2)

similarity_lines.add_hline(y=0, line_dash="dash", line_color="#6B7280")
similarity_lines.update_xaxes(
    title_text="fraction of target training pool (%)", row=1, col=1
)
similarity_lines.update_yaxes(title_text="transfer gain, Δ macro R²", row=1, col=1)
similarity_lines.update_xaxes(
    title_text="mean pairwise static-v0 cosine", row=1, col=2
)
similarity_lines.update_yaxes(title_text="transfer gain, Δ macro R²", row=1, col=2)
similarity_lines.update_layout(
    title="Read similarity and label budget as two-dimensional learning curves",
    template="plotly_white",
    height=680,
    margin={"b": 190},
    legend={
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": -0.16,
        "yanchor": "top",
    },
)
similarity_lines.show()

In [16]:
# %% hide input
efficiency_records = []
for count in sorted(transfer_summary["finger_count"].unique()):
    full = float(full_specialist.query("finger_count == @count")["full_specialist_r2"].iloc[0])
    threshold = 0.98 * full
    for method in ["specialist", "transfer"]:
        column = f"r2_{method}"
        reached = transfer_summary[
            (transfer_summary["finger_count"] == count)
            & (transfer_summary[column] >= threshold)
        ]
        efficiency_records.append({
            "finger_count": count,
            "method": method,
            "required_fraction": (
                float(reached["fraction"].min()) if not reached.empty else np.nan
            ),
            "threshold_r2": threshold,
        })
efficiency = pd.DataFrame(efficiency_records)
figure = go.Figure()
for method, color in [("specialist", "#D1495B"), ("transfer", "#00798C")]:
    frame = efficiency.query("method == @method")
    figure.add_trace(go.Bar(
        x=frame["finger_count"], y=100 * frame["required_fraction"],
        name=method, marker_color=color,
        hovertemplate=(
            f"<b>{method}</b><br>fingers=%{{x}}"
            "<br>required target pool=%{y:.0f}%<extra></extra>"
        ),
    ))
figure.update_layout(
    title="Label budget required to reach 98% of full-specialist R²",
    xaxis_title="target finger count",
    yaxis_title="target training pool required (%)",
    barmode="group", template="plotly_white", height=540,
)
figure.show()

## 5. Study 2: use embedding similarity as an applicability gate

In design automation, the important question is not only “what does the model predict?” but “when should we
trust it instead of launching another expensive Q3D solve?”

For every non-source design, we compute cosine similarity to the normalized foundation centroid and compare it
with the foundation's zero-shot error. A useful applicability score should rank high-error designs toward the
low-similarity end. This enables a hybrid workflow:

> predict high-similarity candidates immediately; route low-similarity candidates to simulation and add them
> to the next adaptation batch.

In [17]:
zero_shot_predictions = np.stack([
    model.predict(model_features)
    for model in foundation_models
])
zero_shot_ape = 100 * np.mean(
    np.abs(zero_shot_predictions - targets[None, :, :])
    / np.maximum(np.abs(targets[None, :, :]), 1e-9),
    axis=2,
)
data["zero_shot_ape_percent"] = zero_shot_ape.mean(axis=0)
target_mask = data["finger_count"] != BASE_FINGER_COUNT
target_data = data.loc[target_mask].copy()
target_data["similarity_decile"] = pd.qcut(
    target_data["similarity_to_base"], q=10,
    labels=[f"D{index}" for index in range(1, 11)],
)
deciles = target_data.groupby("similarity_decile", observed=True).agg(
    similarity=("similarity_to_base", "mean"),
    median_ape=("zero_shot_ape_percent", "median"),
    p90_ape=("zero_shot_ape_percent", lambda values: values.quantile(0.90)),
    designs=("design_id", "size"),
).reset_index()
target_indices = np.flatnonzero(target_mask)
similarity_correlations = np.asarray([
    spearmanr(
        data.loc[target_mask, "similarity_to_base"],
        zero_shot_ape[repeat, target_indices],
    ).statistic
    for repeat in range(N_REPEATS)
])
similarity_error_correlation = {
    "mean": similarity_correlations.mean(),
    "ci95": ci95(similarity_correlations),
}
coverage_records = []
ordered = target_data.sort_values("similarity_to_base", ascending=False)
for coverage in np.linspace(0.10, 1.00, 10):
    accepted = ordered.head(round(coverage * len(ordered)))
    coverage_records.append({
        "coverage": coverage,
        "similarity_threshold": accepted["similarity_to_base"].min(),
        "median_ape": accepted["zero_shot_ape_percent"].median(),
        "p90_ape": accepted["zero_shot_ape_percent"].quantile(0.90),
    })
coverage_curve = pd.DataFrame(coverage_records)
similarity_error_correlation


{'mean': -0.3431092026130833, 'ci95': 0.04457205668926081}

In [18]:
# %% hide input
applicability = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Row-level similarity versus zero-shot error",
        "Error by similarity decile",
        "Accuracy-coverage tradeoff",
        "Domain-level trust map",
    ],
)
sample = target_data.sample(min(6_000, len(target_data)), random_state=RANDOM_SEED)
applicability.add_trace(go.Scattergl(
    x=sample["similarity_to_base"],
    y=sample["zero_shot_ape_percent"].clip(upper=200),
    mode="markers", marker={"size": 4, "opacity": 0.25, "color": "#6A4C93"},
    name="designs",
    hovertemplate="similarity=%{x:.3f}<br>mean APE=%{y:.1f}%<extra></extra>",
), row=1, col=1)
applicability.add_trace(go.Scatter(
    x=deciles["similarity"], y=deciles["median_ape"],
    mode="lines+markers", marker={"size": 8}, line={"width": 3, "color": "#00798C"},
    name="decile median",
), row=1, col=1)
applicability.add_trace(go.Bar(
    x=deciles["similarity_decile"], y=deciles["median_ape"],
    marker_color="#E9C46A", name="median APE",
), row=1, col=2)
applicability.add_trace(go.Scatter(
    x=deciles["similarity_decile"], y=deciles["p90_ape"],
    mode="lines+markers", line={"color": "#D1495B", "width": 3},
    name="90th percentile APE",
), row=1, col=2)
applicability.add_trace(go.Scatter(
    x=100 * coverage_curve["coverage"], y=coverage_curve["median_ape"],
    mode="lines+markers", line={"color": "#00798C", "width": 3},
    name="accepted median APE",
), row=2, col=1)
applicability.add_trace(go.Scatter(
    x=100 * coverage_curve["coverage"], y=coverage_curve["p90_ape"],
    mode="lines+markers", line={"color": "#D1495B", "width": 3},
    name="accepted p90 APE",
), row=2, col=1)
domain_trust = target_data.groupby("finger_count").agg(
    similarity=("similarity_to_base", "mean"),
    median_ape=("zero_shot_ape_percent", "median"),
    designs=("design_id", "size"),
).reset_index()
applicability.add_trace(go.Scatter(
    x=domain_trust["similarity"], y=domain_trust["median_ape"],
    mode="markers+text", text=domain_trust["finger_count"],
    textposition="top center",
    marker={
        "size": np.sqrt(domain_trust["designs"]),
        "color": domain_trust["finger_count"],
        "colorscale": "Turbo", "showscale": True,
        "colorbar": {
            "title": "fingers",
            "x": 1.01,
            "xanchor": "left",
            "y": 0.21,
            "len": 0.36,
        },
    },
    name="finger-count domains",
), row=2, col=2)
applicability.update_xaxes(title_text="cosine to foundation centroid", row=1, col=1)
applicability.update_yaxes(title_text="mean absolute percentage error (%)", row=1, col=1)
applicability.update_xaxes(title_text="similarity decile (low → high)", row=1, col=2)
applicability.update_yaxes(title_text="absolute percentage error (%)", row=1, col=2)
applicability.update_xaxes(title_text="designs accepted without simulation (%)", row=2, col=1)
applicability.update_yaxes(title_text="accepted-set error (%)", row=2, col=1)
applicability.update_xaxes(title_text="mean cosine to foundation", row=2, col=2)
applicability.update_yaxes(title_text="median zero-shot APE (%)", row=2, col=2)
applicability.update_layout(
    title=(
        "Similarity is an actionable applicability score "
        f"(mean Spearman ρ={similarity_error_correlation['mean']:.3f} "
        f"± {similarity_error_correlation['ci95']:.3f}, 95% CI half-width)"
    ),
    template="plotly_white", height=900,
    margin={"t": 115, "r": 180, "b": 115},
    legend={
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": -0.08,
        "yanchor": "top",
    },
)
applicability.show()

## 6. What the repeated experiment establishes

- The expanded dataset covers 13 exact finger-count domains and 17 varying geometry parameters.
- The ablation shows that geometry carries the predictive signal: at 10% labels the full representation
  reaches mean domain-macro R² = 0.892, compared with 0.888 without domain identity, 0.835 from scalar
  geometry, 0.622 from shape alone, and approximately 0 from finger-count identity alone.
- Transfer is **heterogeneous**, not universally beneficial. We therefore mark only paired effects whose 95%
  interval excludes zero and whose Benjamini–Hochberg-adjusted `q < 0.05`.
- Across 12 independently seeded foundations, embedding similarity and zero-shot error have mean Spearman
  ρ = -0.354 with a 95% interval half-width of 0.033. Similarity is useful as a routing prior, but not as a
  standalone guarantee.

Repeated holdout reduces dependence on one favorable partition, but overlapping holdouts are not twelve
independent physical experiments. The correct conclusion is evidence of robustness within this simulated
catalogue, followed by a need for external validation on new simulations or fabricated-device measurements.


Because Tutorial 16b equalizes domain size before all processing, differences between finger-count learning curves cannot be attributed to the approximately 1,670 versus 1,260 raw-row imbalance in the source release. They instead reflect the sampled geometry/capacitance distributions and model behavior under equal data budgets.

## 7. Sweep every finger count as the foundation

The repeated count-8 study answers one source-to-many-targets question. We now repeat the same controlled experiment
with foundation finger counts 2, 3, ..., 14 on every independent holdout. For every ordered source-target pair we:

1. fit the foundation on the source domain's complete training pool;
2. adapt it with every integer percentage from 1% through 100% of the target pool, matching the grid above;
3. compare against the corresponding specialist from the same repeat and label budget;
4. retain source-target static-v0 cosine similarity as an entirely label-free applicability signal.

The diagonal is omitted because transferring a foundation back to its own training domain is not a
cross-domain experiment. This larger model bank has its own fingerprinted checkpoint, so rerunning the
tutorial restores the trained weights unless its data or settings change.

In [19]:
SWEEP_CONFIG = {
    "checkpoint_version": 3,
    "source_finger_counts": list(range(2, 15)),
    "fractions": FRACTIONS,
    "repeats": MODEL_CONFIG["repeats"],
    "split_strategy": MODEL_CONFIG["split_strategy"],
    "confidence_level": CONFIDENCE_LEVEL,
    "alpha": ALPHA,
    "experiment_id": EXPERIMENT_ID,
}
sweep_fingerprint = hashlib.sha256(
    json.dumps(SWEEP_CONFIG, sort_keys=True).encode()
).hexdigest()[:16]
SWEEP_CHECKPOINT_DIR = CHECKPOINT_DIR / f"all-foundations-{sweep_fingerprint}"
SWEEP_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SWEEP_METRICS_PATH = SWEEP_CHECKPOINT_DIR / "transfer-curves.parquet"
SWEEP_WEIGHTS_PATH = SWEEP_CHECKPOINT_DIR / "trained-weights.npz"
SWEEP_INDEX_PATH = SWEEP_CHECKPOINT_DIR / "model-index.json"
SWEEP_METADATA_PATH = SWEEP_CHECKPOINT_DIR / "metadata.json"

sweep_checkpoint_complete = all(path.exists() for path in [
    SWEEP_METRICS_PATH, SWEEP_WEIGHTS_PATH, SWEEP_INDEX_PATH, SWEEP_METADATA_PATH,
])
if sweep_checkpoint_complete:
    sweep_curves = pd.read_parquet(SWEEP_METRICS_PATH)
    sweep_trained_weights = np.load(SWEEP_WEIGHTS_PATH)["weights"]
    sweep_model_index = pd.DataFrame(json.loads(SWEEP_INDEX_PATH.read_text()))
    sweep_checkpoint_state = (
        f"Loaded {len(sweep_model_index):,} all-foundation models from checkpoint."
    )
else:
    specialist_lookup = curves.query("method == 'specialist'").set_index(
        ["finger_count", "fraction", "repeat"]
    )
    sweep_weight_bank = []
    sweep_model_records = []
    sweep_records = []

    def save_sweep_model(model_key, model, **fields):
        sweep_model_records.append({
            "model_key": model_key,
            "weight_index": len(sweep_weight_bank),
            "alpha": model.alpha,
            **fields,
        })
        sweep_weight_bank.append(model.weights_.copy())

    for repeat in range(MODEL_CONFIG["repeats"]):
        repeat_splits = splits_by_repeat[repeat]
        orders = nested_training_indices(repeat)
        for source_count in range(2, 15):
            source_pool = repeat_splits[source_count]["pool"]
            source_foundation = TransferRidgeRegressor(ALPHA).fit(
                model_features[source_pool], targets[source_pool]
            )
            save_sweep_model(
                f"foundation-c{source_count}-r{repeat}",
                source_foundation, method="foundation",
                source_finger_count=source_count, repeat=repeat,
                training_rows=len(source_pool),
            )
            zero_shot_by_target = {}
            for target_count in range(2, 15):
                if target_count == source_count:
                    continue
                target_test = repeat_splits[target_count]["test"]
                zero_shot_by_target[target_count] = macro_metrics(
                    targets[target_test],
                    source_foundation.predict(model_features[target_test]),
                )

            for fraction in FRACTIONS:
                fraction_key = f"{fraction:.2f}"
                for target_count in range(2, 15):
                    if target_count == source_count:
                        continue
                    selected = orders[target_count][
                        :max(1, round(fraction * len(orders[target_count])))
                    ]
                    target_test = repeat_splits[target_count]["test"]
                    adapted = TransferRidgeRegressor(ALPHA).fit(
                        model_features[selected], targets[selected],
                        prior=source_foundation,
                    )
                    save_sweep_model(
                        (
                            f"transfer-s{source_count}-t{target_count}"
                            f"-r{repeat}-f{fraction_key}"
                        ),
                        adapted, method="transfer",
                        source_finger_count=source_count,
                        target_finger_count=target_count,
                        repeat=repeat, fraction=fraction,
                        training_rows=len(selected),
                    )
                    transfer_score = macro_metrics(
                        targets[target_test],
                        adapted.predict(model_features[target_test]),
                    )
                    specialist_score = specialist_lookup.loc[
                        (target_count, fraction, repeat)
                    ]
                    zero_shot_score = zero_shot_by_target[target_count]
                    sweep_records.append({
                        "source_finger_count": source_count,
                        "target_finger_count": target_count,
                        "finger_distance": abs(source_count - target_count),
                        "fraction": fraction, "repeat": repeat,
                        "labels": len(selected),
                        "mean_pairwise_cosine": float(
                            raw_means[source_count] @ raw_means[target_count]
                        ),
                        "r2_zero_shot": float(zero_shot_score["r2"]),
                        "r2_transfer": float(transfer_score["r2"]),
                        "r2_specialist": float(specialist_score["r2"]),
                        "r2_gain": float(
                            transfer_score["r2"] - specialist_score["r2"]
                        ),
                        "mae_transfer_fF": float(transfer_score["mae"]),
                        "mae_specialist_fF": float(specialist_score["mae"]),
                        "mae_reduction_fF": float(
                            specialist_score["mae"] - transfer_score["mae"]
                        ),
                        "within_5_percent_gain": float(
                            transfer_score["within_5_percent"]
                            - specialist_score["within_5_percent"]
                        ),
                    })

    sweep_curves = pd.DataFrame(sweep_records)
    sweep_trained_weights = np.stack(sweep_weight_bank)
    sweep_model_index = pd.DataFrame(sweep_model_records)
    sweep_curves.to_parquet(SWEEP_METRICS_PATH, index=False)
    with SWEEP_WEIGHTS_PATH.open("wb") as checkpoint_file:
        np.savez_compressed(checkpoint_file, weights=sweep_trained_weights)
    SWEEP_INDEX_PATH.write_text(json.dumps(
        json.loads(sweep_model_index.to_json(orient="records")), indent=2
    ))
    SWEEP_METADATA_PATH.write_text(json.dumps({
        "sweep_fingerprint": sweep_fingerprint,
        "config": SWEEP_CONFIG,
        "models": len(sweep_model_index),
        "records": len(sweep_curves),
    }, indent=2))
    sweep_checkpoint_state = (
        f"Trained and checkpointed {len(sweep_model_index):,} "
        "all-foundation models."
    )

sweep_summary = sweep_curves.groupby(
    [
        "source_finger_count", "target_finger_count",
        "finger_distance", "fraction",
    ],
    as_index=False,
).agg(
    mean_pairwise_cosine=("mean_pairwise_cosine", "first"),
    r2_zero_shot=("r2_zero_shot", "mean"),
    r2_transfer=("r2_transfer", "mean"),
    r2_specialist=("r2_specialist", "mean"),
    r2_gain=("r2_gain", "mean"),
    r2_gain_ci95=("r2_gain", ci95),
    mae_reduction_fF=("mae_reduction_fF", "mean"),
    within_5_percent_gain=("within_5_percent_gain", "mean"),
    labels=("labels", "mean"),
)
sweep_summary["r2_gain_lower"] = (
    sweep_summary["r2_gain"] - sweep_summary["r2_gain_ci95"]
)
sweep_summary["r2_gain_upper"] = (
    sweep_summary["r2_gain"] + sweep_summary["r2_gain_ci95"]
)
sweep_tests = sweep_curves.groupby(
    ["source_finger_count", "target_finger_count", "fraction"]
)["r2_gain"].apply(
    lambda values: ttest_1samp(values, 0.0).pvalue
).reset_index(name="p_value")
sweep_tests["q_value"] = bh_adjust(sweep_tests["p_value"])
sweep_summary = sweep_summary.merge(
    sweep_tests,
    on=["source_finger_count", "target_finger_count", "fraction"],
)
sweep_summary["robust_gain"] = (
    (sweep_summary["q_value"] < 0.05)
    & (
        (sweep_summary["r2_gain_lower"] > 0)
        | (sweep_summary["r2_gain_upper"] < 0)
    )
)
print(sweep_checkpoint_state)
print(f"Checkpoint: {SWEEP_CHECKPOINT_DIR}")
pd.DataFrame({
    "quantity": [
        "independent holdouts", "foundation domains",
        "ordered source-target pairs", "adapted models",
        "source-target-fraction summaries",
    ],
    "value": [
        MODEL_CONFIG["repeats"], 13, 13 * 12,
        len(sweep_model_index.query("method == 'transfer'")),
        len(sweep_summary),
    ],
})


Loaded 187,356 all-foundation models from checkpoint.
Checkpoint: /Users/shanto/.cache/squadds/tutorial16b/18005b4aa44ee1a4581f/all-foundations-9f3ab1ebea2e7fd0


,quantity,value
0,independent holdouts,12
1,foundation domains,13
2,ordered source-target pairs,156
3,adapted models,187200
4,source-target-fraction summaries,15600


### 7.1 See the source-target geometry before reading model scores

The left matrix is entirely unsupervised: it contains only mean pairwise static-v0 cosine similarity. The
right matrix averages transfer gain across the complete low-budget regime from 1% through 10%. Reading the matrices
together exposes both **locality** and **directionality**: similarity is symmetric, but transfer need not be,
because the source foundation and target adaptation sets play different roles.

In [20]:
# %% hide input
finger_range = list(range(2, 15))
similarity_matrix = pd.DataFrame(
    [
        [
            float(raw_means[source] @ raw_means[target])
            for source in finger_range
        ]
        for target in finger_range
    ],
    index=finger_range,
    columns=finger_range,
)
low_budget_gain = sweep_summary.query(
    "fraction <= 0.10"
).groupby(
    ["target_finger_count", "source_finger_count"]
)["r2_gain"].mean().unstack()

atlas = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Static-v0 cosine similarity",
        "Mean transfer gain at 1–10% target labels",
    ],
    horizontal_spacing=0.12,
)
atlas.add_trace(go.Heatmap(
    z=similarity_matrix.to_numpy(),
    x=similarity_matrix.columns,
    y=similarity_matrix.index,
    colorscale="Viridis",
    zmin=float(similarity_matrix.to_numpy().min()),
    zmax=1.0,
    showscale=False,
    text=np.round(similarity_matrix.to_numpy(), 2),
    texttemplate="%{text}",
    hovertemplate=(
        "foundation=%{x} fingers<br>target=%{y} fingers"
        "<br>mean pairwise cosine=%{z:.4f}<extra></extra>"
    ),
), row=1, col=1)
atlas.add_trace(go.Heatmap(
    z=low_budget_gain.reindex(
        index=finger_range, columns=finger_range
    ).to_numpy(),
    x=finger_range,
    y=finger_range,
    colorscale="RdBu",
    zmid=0,
    colorbar={
        "title": "Δ macro R²",
        "x": 1.02,
        "xanchor": "left",
        "len": 0.78,
    },
    text=np.round(low_budget_gain.reindex(
        index=finger_range, columns=finger_range
    ).to_numpy(), 3),
    texttemplate="%{text}",
    hovertemplate=(
        "foundation=%{x} fingers<br>target=%{y} fingers"
        "<br>low-budget transfer ΔR²=%{z:.4f}<extra></extra>"
    ),
), row=1, col=2)
atlas.update_xaxes(title_text="foundation finger count")
atlas.update_yaxes(title_text="target finger count")
atlas.update_yaxes(autorange="reversed")
atlas.update_layout(
    title="Foundation applicability atlas: similarity is symmetric; efficacy is directional",
    template="plotly_white",
    height=660,
    margin={"r": 145},
)
atlas.show()

### 7.2 Rank foundations without hiding the target domains

For each foundation and label budget, the first panel averages transfer gain equally over all 12 target
domains. The second panel reports the fraction of those targets where transfer beats a specialist trained on
the same labels. A strong basis should raise both the average gain and the positive-transfer coverage.

In [21]:
# %% hide input
source_profile_repeats = sweep_curves.assign(
    positive_transfer=sweep_curves["r2_gain"] > 0
).groupby(
    ["source_finger_count", "fraction", "repeat"], as_index=False
).agg(
    mean_r2_gain=("r2_gain", "mean"),
    positive_transfer_rate=("positive_transfer", "mean"),
)
source_profiles = source_profile_repeats.groupby(
    ["source_finger_count", "fraction"], as_index=False
).agg(
    mean_r2_gain=("mean_r2_gain", "mean"),
    mean_r2_gain_ci95=("mean_r2_gain", ci95),
    positive_transfer_rate=("positive_transfer_rate", "mean"),
    positive_transfer_rate_ci95=("positive_transfer_rate", ci95),
)
source_profiles = source_profiles.merge(
    sweep_summary.groupby(
        ["source_finger_count", "fraction"], as_index=False
    ).agg(
        median_r2_gain=("r2_gain", "median"),
        mean_cosine=("mean_pairwise_cosine", "mean"),
    ),
    on=["source_finger_count", "fraction"],
)
profile_fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 1.00]
profile_colors = [
    "#D1495B", "#F4A261", "#E9C46A", "#2A9D8F",
    "#00798C", "#6A4C93", "#264653",
]
profiles = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Average efficacy over all target domains",
        "How often transfer beats the matched specialist",
    ],
    horizontal_spacing=0.12,
)
for fraction, color in zip(profile_fractions, profile_colors):
    frame = source_profiles.query("fraction == @fraction").sort_values(
        "source_finger_count"
    )
    common = np.column_stack([frame["mean_cosine"], frame["median_r2_gain"]])
    profiles.add_trace(go.Scatter(
        x=frame["source_finger_count"], y=frame["mean_r2_gain"],
        error_y={
            "type": "data", "array": frame["mean_r2_gain_ci95"], "visible": True,
        },
        mode="lines+markers", name=f"{100 * fraction:.0f}% target pool",
        line={"color": color, "width": 3}, marker={"size": 8},
        customdata=common,
        hovertemplate=(
            "foundation=%{x} fingers<br>mean ΔR²=%{y:.4f}"
            "<br>median ΔR²=%{customdata[1]:.4f}"
            "<br>mean target cosine=%{customdata[0]:.4f}<extra></extra>"
        ),
    ), row=1, col=1)
    profiles.add_trace(go.Scatter(
        x=frame["source_finger_count"],
        y=100 * frame["positive_transfer_rate"],
        error_y={
            "type": "data",
            "array": 100 * frame["positive_transfer_rate_ci95"],
            "visible": True,
        },
        mode="lines+markers", showlegend=False,
        line={"color": color, "width": 3}, marker={"size": 8},
        hovertemplate=(
            "foundation=%{x} fingers"
            "<br>targets helped=%{y:.1f}%<extra></extra>"
        ),
    ), row=1, col=2)
profiles.add_hline(y=0, line_dash="dash", line_color="#6B7280", row=1, col=1)
profiles.update_xaxes(title_text="foundation finger count")
profiles.update_yaxes(title_text="mean transfer gain, Δ macro R²", row=1, col=1)
profiles.update_yaxes(
    title_text="target domains with positive transfer (%)",
    range=[0, 105], row=1, col=2,
)
profiles.update_layout(
    title="Foundation efficacy and applicability (mean ± 95% repeated-holdout CI)",
    template="plotly_white", height=620, margin={"b": 125},
    legend={
        "orientation": "h", "x": 0.5, "xanchor": "center",
        "y": -0.10, "yanchor": "top",
    },
)
profiles.show()


### 7.3 Turn cosine similarity into an interpretable line plot

To remove source-count noise, we group all ordered source-target pairs into six cosine-similarity bands. The
left panel asks whether increasingly similar geometry improves transfer. The right panel provides the
physical cross-check: finger-count distance should generally move in the opposite direction. Error bars show
95% confidence intervals across independent holdouts after averaging within each source-target group.

In [22]:
# %% hide input
relationship_records = []
distance_records = []
for fraction in profile_fractions:
    frame = sweep_summary.query("fraction == @fraction").copy()
    frame["similarity_band"] = pd.qcut(
        frame["mean_pairwise_cosine"],
        q=6,
        duplicates="drop",
    )
    similarity_groups = frame.groupby(
        "similarity_band", observed=True
    ).agg(
        cosine=("mean_pairwise_cosine", "mean"),
        gain=("r2_gain", "mean"),
        gain_std=("r2_gain", "std"),
        pairs=("r2_gain", "size"),
    ).reset_index(drop=True)
    similarity_groups["gain_sem"] = (
        similarity_groups["gain_std"] / np.sqrt(similarity_groups["pairs"])
    )
    similarity_groups["fraction"] = fraction
    relationship_records.append(similarity_groups)

    distance_groups = frame.groupby("finger_distance").agg(
        gain=("r2_gain", "mean"),
        gain_std=("r2_gain", "std"),
        pairs=("r2_gain", "size"),
    ).reset_index()
    distance_groups["gain_sem"] = (
        distance_groups["gain_std"] / np.sqrt(distance_groups["pairs"])
    )
    distance_groups["fraction"] = fraction
    distance_records.append(distance_groups)

similarity_relationship = pd.concat(relationship_records, ignore_index=True)
distance_relationship = pd.concat(distance_records, ignore_index=True)
relationship = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Higher static-v0 similarity",
        "Larger finger-count separation",
    ],
    horizontal_spacing=0.12,
)
for fraction, color in zip(profile_fractions, profile_colors):
    cosine_frame = similarity_relationship.query(
        "fraction == @fraction"
    ).sort_values("cosine")
    relationship.add_trace(go.Scatter(
        x=cosine_frame["cosine"],
        y=cosine_frame["gain"],
        error_y={
            "type": "data",
            "array": cosine_frame["gain_sem"],
            "visible": True,
            "thickness": 1.2,
        },
        mode="lines+markers",
        name=f"{100 * fraction:.0f}% target pool",
        line={"color": color, "width": 3},
        marker={"size": 8},
        hovertemplate=(
            "mean cosine=%{x:.4f}<br>mean transfer ΔR²=%{y:.4f}"
            "<extra></extra>"
        ),
    ), row=1, col=1)
    distance_frame = distance_relationship.query(
        "fraction == @fraction"
    ).sort_values("finger_distance")
    relationship.add_trace(go.Scatter(
        x=distance_frame["finger_distance"],
        y=distance_frame["gain"],
        error_y={
            "type": "data",
            "array": distance_frame["gain_sem"],
            "visible": True,
            "thickness": 1.2,
        },
        mode="lines+markers",
        name=f"{100 * fraction:.0f}% target pool",
        showlegend=False,
        line={"color": color, "width": 3},
        marker={"size": 8},
        hovertemplate=(
            "finger-count distance=%{x:.0f}"
            "<br>mean transfer ΔR²=%{y:.4f}<extra></extra>"
        ),
    ), row=1, col=2)
relationship.add_hline(y=0, line_dash="dash", line_color="#6B7280")
relationship.update_xaxes(
    title_text="mean pairwise static-v0 cosine", row=1, col=1
)
relationship.update_xaxes(
    title_text="|foundation fingers − target fingers|", row=1, col=2
)
relationship.update_yaxes(title_text="mean transfer gain, Δ macro R²", row=1, col=1)
relationship.update_yaxes(title_text="mean transfer gain, Δ macro R²", row=1, col=2)
relationship.update_layout(
    title="Similarity predicts where a foundation prior is useful",
    template="plotly_white",
    height=620,
    margin={"b": 125},
    legend={
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": -0.10,
        "yanchor": "top",
    },
)
relationship.show()

### 7.4 Convert efficacy into a simulation-budget decision

For each ordered foundation-target pair, the final matrix asks how much labeled target data is required for
transfer to reach 98% of that target's full-specialist R². Green cells are the strongest evidence for useful
transfer: a foundation reaches specialist-level accuracy after seeing only a small target subset. Blank cells
never cross the threshold within the tested budgets.

In [23]:
# %% hide input
required_records = []
full_r2_by_target = full_specialist.set_index("finger_count")[
    "full_specialist_r2"
].to_dict()
for (source_count, target_count), frame in sweep_summary.groupby(
    ["source_finger_count", "target_finger_count"]
):
    threshold = 0.98 * full_r2_by_target[target_count]
    reached = frame.loc[frame["r2_transfer"] >= threshold].sort_values("fraction")
    required_records.append({
        "source_finger_count": source_count,
        "target_finger_count": target_count,
        "required_fraction": (
            float(reached["fraction"].iloc[0]) if not reached.empty else np.nan
        ),
        "mean_pairwise_cosine": float(frame["mean_pairwise_cosine"].iloc[0]),
    })
required_by_foundation = pd.DataFrame(required_records)
required_matrix = required_by_foundation.pivot(
    index="target_finger_count",
    columns="source_finger_count",
    values="required_fraction",
).reindex(index=finger_range, columns=finger_range)
required_values = required_matrix.to_numpy()
required_text = np.full(required_values.shape, "", dtype=object)
required_valid = np.isfinite(required_values)
required_text[required_valid] = [
    f"{100 * value:.0f}%"
    for value in required_values[required_valid]
]
figure = go.Figure(go.Heatmap(
    z=100 * required_matrix.to_numpy(),
    x=required_matrix.columns,
    y=required_matrix.index,
    colorscale="RdYlGn_r",
    zmin=1,
    zmax=100,
    colorbar={
        "title": "target pool required",
        "ticksuffix": "%",
        "x": 1.02,
        "xanchor": "left",
    },
    text=required_text,
    texttemplate="%{text}",
    customdata=np.stack([
        np.tile(required_matrix.columns.to_numpy(), (len(required_matrix), 1)),
        np.tile(required_matrix.index.to_numpy()[:, None], (1, len(required_matrix.columns))),
    ], axis=-1),
    hovertemplate=(
        "foundation=%{customdata[0]} fingers"
        "<br>target=%{customdata[1]} fingers"
        "<br>target pool required=%{z:.0f}%<extra></extra>"
    ),
))
figure.update_layout(
    title="Target labels required to reach 98% of full-specialist R²",
    xaxis_title="foundation finger count",
    yaxis_title="target finger count",
    yaxis_autorange="reversed",
    template="plotly_white",
    height=680,
    margin={"r": 170},
)
figure.show()

### 7.5 Conclusions from changing the foundation basis

- Foundation choice measurably changes transfer efficacy, but many individual source-target-budget effects
  remain inconclusive after paired confidence intervals and false-discovery-rate correction.
- Static-v0 cosine similarity is an unsupervised applicability prior, not a proof that transfer will help.
- Directional cases with similar cosine can behave differently, so deployment should combine similarity with
  held-out adaptation metrics and report uncertainty.
- A practical workflow is to rank candidate foundations by similarity, validate candidates on a small target
  subset, and expand simulation whenever the adjusted transfer interval does not clear zero.


### 7.6 Final interactive foundation snapshot

Use the slider below to replace the foundation model while keeping all four views synchronized. The dashboard
is a compact reading order for a candidate basis:

1. **similarity profile**: which target geometries resemble this foundation without using labels;
2. **low-budget gain**: where that resemblance actually helps across 1–10% target data;
3. **learning curves**: whether help persists or vanishes as target labels accumulate;
4. **required target pool**: the simulation budget needed to reach 98% of full-specialist R².

In [24]:
# %% hide input
def foundation_snapshot(source_count):
    similarity = similarity_matrix[source_count].reindex(finger_range)
    low_gain = low_budget_gain[source_count].reindex(finger_range)
    required = required_matrix[source_count].reindex(finger_range)
    gain_curves = []
    for target_count in finger_range:
        frame = sweep_summary.query(
            "source_finger_count == @source_count "
            "and target_finger_count == @target_count"
        ).sort_values("fraction")
        gain_curves.append(
            frame["r2_gain"].to_numpy()
            if not frame.empty
            else np.full(len(FRACTIONS), np.nan)
        )
    low_text = [
        "" if not np.isfinite(value) else f"{value:+.3f}"
        for value in low_gain.to_numpy()
    ]
    required_text = [
        "" if not np.isfinite(value) else f"{100 * value:.0f}%"
        for value in required.to_numpy()
    ]
    return {
        "similarity": similarity.to_numpy(),
        "low_gain": low_gain.to_numpy(),
        "gain_curves": gain_curves,
        "required": 100 * required.to_numpy(),
        "low_text": low_text,
        "required_text": required_text,
    }


initial_source = BASE_FINGER_COUNT
initial_snapshot = foundation_snapshot(initial_source)
snapshot = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Unsupervised target similarity",
        "Transfer gain at 1–10% target data",
        "Transfer gain as target labels increase",
        "Target data needed for 98% of specialist R²",
    ],
    horizontal_spacing=0.11,
    vertical_spacing=0.16,
)
snapshot.add_trace(go.Scatter(
    x=finger_range,
    y=initial_snapshot["similarity"],
    mode="lines+markers",
    line={"color": "#6A4C93", "width": 3},
    marker={"size": 9},
    showlegend=False,
    customdata=finger_range,
    hovertemplate=(
        "target=%{customdata} fingers"
        "<br>mean pairwise cosine=%{y:.4f}<extra></extra>"
    ),
), row=1, col=1)
snapshot.add_trace(go.Bar(
    x=finger_range,
    y=initial_snapshot["low_gain"],
    text=initial_snapshot["low_text"],
    textposition="outside",
    marker_color="#00798C",
    showlegend=False,
    hovertemplate=(
        "target=%{x} fingers"
        "<br>low-budget transfer ΔR²=%{y:.4f}<extra></extra>"
    ),
), row=1, col=2)
for target_count, gain_curve in zip(
    finger_range, initial_snapshot["gain_curves"]
):
    snapshot.add_trace(go.Scatter(
        x=100 * np.asarray(FRACTIONS),
        y=gain_curve,
        mode="lines+markers",
        name=f"target {target_count}",
        line={"color": colors[target_count], "width": 2.2},
        marker={"size": 6},
        hovertemplate=(
            f"<b>target: {target_count} fingers</b><br>"
            "target pool=%{x:.0f}%<br>transfer ΔR²=%{y:.4f}<extra></extra>"
        ),
    ), row=2, col=1)
snapshot.add_trace(go.Bar(
    x=finger_range,
    y=initial_snapshot["required"],
    text=initial_snapshot["required_text"],
    textposition="outside",
    marker_color="#E9C46A",
    showlegend=False,
    hovertemplate=(
        "target=%{x} fingers"
        "<br>target pool required=%{y:.0f}%<extra></extra>"
    ),
), row=2, col=2)

slider_steps = []
for source_count in finger_range:
    source_snapshot = foundation_snapshot(source_count)
    updated_y = [
        source_snapshot["similarity"],
        source_snapshot["low_gain"],
        *source_snapshot["gain_curves"],
        source_snapshot["required"],
    ]
    updated_text = [
        None,
        source_snapshot["low_text"],
        *([None] * len(finger_range)),
        source_snapshot["required_text"],
    ]
    slider_steps.append({
        "label": str(source_count),
        "method": "update",
        "args": [
            {"y": updated_y, "text": updated_text},
            {
                "title.text": (
                    f"Foundation snapshot · {source_count} fingers"
                )
            },
        ],
    })

snapshot.add_hline(
    y=0, line_dash="dash", line_color="#6B7280", row=1, col=2
)
snapshot.add_hline(
    y=0, line_dash="dash", line_color="#6B7280", row=2, col=1
)
snapshot.update_xaxes(title_text="target finger count", row=1, col=1)
snapshot.update_yaxes(title_text="mean pairwise cosine", row=1, col=1)
snapshot.update_xaxes(title_text="target finger count", row=1, col=2)
snapshot.update_yaxes(title_text="transfer gain, Δ macro R²", row=1, col=2)
snapshot.update_xaxes(title_text="target training pool used (%)", row=2, col=1)
snapshot.update_yaxes(title_text="transfer gain, Δ macro R²", row=2, col=1)
snapshot.update_xaxes(title_text="target finger count", row=2, col=2)
snapshot.update_yaxes(
    title_text="target pool required (%)", range=[0, 112], row=2, col=2
)
snapshot.update_layout(
    title=f"Foundation snapshot · {initial_source} fingers",
    template="plotly_white",
    height=980,
    margin={"b": 235},
    legend={
        "title": "learning-curve target",
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": -0.08,
        "yanchor": "top",
    },
    sliders=[{
        "active": finger_range.index(initial_source),
        "steps": slider_steps,
        "x": 0.05,
        "len": 0.90,
        "y": -0.24,
        "currentvalue": {
            "prefix": "Foundation finger count: ",
            "font": {"size": 16},
        },
        "pad": {"t": 55},
    }],
)
snapshot.show()